## 연습 1: 카탈로그 구조 설정

당신은 **Lakeside University**의 데이터 엔지니어로, 학생 기록, 과정 카탈로그 및 등록 데이터를 관리하기 위해 Azure Databricks 및 Unity Catalog에서 데이터 플랫폼을 구축하고 있습니다.

이 연습에서는 최상위 네임스페이스 구조 — 하나의 `catalog`와 **medallion architecture** 명명 패턴을 따르는 세 개의 `schemas`을 만듭니다.

다음은 생성할 전체 Unity Catalog 구조입니다:

```txt

Unity Catalog
└── edu_dev  [catalog]
    │   태그: environment=development | university=lakeside | data_classification=internal
    │
    ├── bronze  [schema]
    │   └── raw_files/  [관리 볼륨]
    │       └── students.csv
    │
    ├── silver  [schema]
    │   ├── students  [테이블]
    │   │   ├── student_id       BIGINT  NOT NULL  ← PK
    │   │   ├── first_name       STRING
    │   │   ├── last_name        STRING
    │   │   ├── email            STRING
    │   │   ├── enrollment_year  INT
    │   │   ├── program          STRING
    │   │   └── phone_number     STRING  (연습 6에서 추가)
    │   │
    │   ├── courses  [테이블]
    │   │   ├── course_id    BIGINT  NOT NULL  ← PK
    │   │   ├── course_name  STRING
    │   │   ├── department   STRING
    │   │   └── credits      INT
    │   │
    │   ├── enrollments  [테이블]
    │   │   ├── enrollment_id  BIGINT  NOT NULL  ← PK
    │   │   ├── student_id     BIGINT            → FK → students.student_id
    │   │   ├── course_id      BIGINT            → FK → courses.course_id
    │   │   ├── semester       STRING
    │   │   └── grade          DECIMAL(4,2)
    │   │
    │   ├── vw_student_enrollments  [보기]
    │   │   └── students + courses + enrollments를 조인합니다
    │   │
    │   └── get_grade_classification(grade)  [함수]
    │       └── STRING 반환  (A / B / C / D / F)
    │
    └── gold  [schema]
        └── vw_department_enrollment_stats  [구체화된 보기]
            └── enrollments + courses를 집계합니다
                (department, total_enrollments, avg_grade, distinct_students)
```

### 작업 1.1 — 개발 카탈로그 생성

설명 주석이 있는 `edu_dev`라는 카탈로그를 만듭니다. 이 카탈로그는 Lakeside University 데이터 플랫폼의 **개발 환경**입니다.

**명명 규칙 리마인더:**
- 소문자 및 언더스코어 사용
- 마침표, 공백, 슬래시 또는 제어 문자 없음

> 🤖 **Genie Code 팁:** Genie Code에 질문하세요:
> *"Databricks SQL에서 주석이 있는 Unity Catalog 카탈로그를 만드는 방법은 무엇입니까?"*

**힌트:** `CREATE CATALOG IF NOT EXISTS` 다음에 `COMMENT` 문자열을 사용합니다.

In [ ]:
# Since no default storage is enabled, we are inheriting the storage path from the default catalog's root.
catalogs = [catalog.name for catalog in spark.catalog.listCatalogs("adb_dp750*")]
dp750_default_catalog = catalogs[0] if catalogs else None

storage_root = (
    spark.sql(f"DESCRIBE CATALOG EXTENDED {dp750_default_catalog}")
    .filter("info_name = 'Storage Root'")
    .select("info_value")
    .first()[0]
)
print (f"Storage root: {storage_root}")

spark.sql(f"""
    CREATE CATALOG IF NOT EXISTS edu_dev
    MANAGED LOCATION '{storage_root}'
    COMMENT 'Development environment for Lakeside University data platform'
""")

### 작업 1.2 — Medallion 스키마 생성

`edu_dev` 카탈로그 내에서 **medallion architecture**를 구현하는 세 개의 스키마를 만듭니다:

| 스키마 | 목적 |
|--------|---------|
| `bronze` | 원본 수집 데이터 — 수정되지 않은 소스 파일 |
| `silver` | 정리, 검증 및 강화된 데이터 |
| `gold` | 집계된, 분석 준비 완료 데이터 |

> 🤖 **Genie Code 팁:** 질문하세요:
> *"Databricks Unity Catalog 카탈로그에서 SQL을 사용하여 여러 스키마를 만드는 방법을 보여주세요."*

**힌트:** 각 스키마에 `CREATE SCHEMA IF NOT EXISTS catalog.schema_name`을 사용합니다.

In [ ]:
%sql
CREATE SCHEMA IF NOT EXISTS edu_dev.bronze
COMMENT 'Raw ingested data — unmodified source files';

CREATE SCHEMA IF NOT EXISTS edu_dev.silver
COMMENT 'Cleaned, validated, and enriched data';

CREATE SCHEMA IF NOT EXISTS edu_dev.gold
COMMENT 'Aggregated, analytics-ready data';

## 연습 2: 제약 조건이 있는 테이블 생성

카탈로그 구조가 준비되었으므로, `silver` 스키마의 Lakeside University 학사 데이터에 대한 핵심 테이블을 만듭니다. **기본 키** 및 **외래 키** 제약 조건을 정의하여 데이터 모델을 문서화합니다. Unity Catalog에서 이러한 제약 조건은 정보성이 있습니다 — 쿼리 최적화에 도움이 되지만 쓰기 시간에 참조 무결성을 적용하지는 않습니다.

### 작업 2.1 — `students` 테이블 생성

다음 열을 포함하여 학생 기록을 저장하는 관리 Delta 테이블 `edu_dev.silver.students`를 만듭니다:

| 열 | 유형 | 설명 |
|--------|------|-------------|
| `student_id` | BIGINT | 고유 학생 식별자 (기본 키) |
| `first_name` | STRING | 학생의 이름 |
| `last_name` | STRING | 학생의 성 |
| `email` | STRING | 대학 이메일 주소 |
| `enrollment_year` | INT | 학생이 등록한 연도 |
| `program` | STRING | 학위 프로그램 |

> 🤖 **Genie Code 팁:** 질문하세요:
> *"Unity Catalog에서 기본 키 제약 조건이 있는 관리 Delta 테이블을 만드는 방법은?"*

**힌트:** 열 목록 끝에 `CONSTRAINT students_pk PRIMARY KEY (student_id)`를 추가합니다.

In [ ]:
%sql
CREATE TABLE IF NOT EXISTS edu_dev.silver.students (
  student_id      BIGINT NOT NULL,
  first_name      STRING,
  last_name       STRING,
  email           STRING,
  enrollment_year INT,
  program         STRING,
  CONSTRAINT students_pk PRIMARY KEY (student_id)
);

### 작업 2.2 — `courses` 테이블 생성

다음 열을 사용하여 `edu_dev.silver.courses`를 만듭니다:

| 열 | 유형 | 설명 |
|--------|------|-------------|
| `course_id` | BIGINT | 고유 과정 식별자 (기본 키) |
| `course_name` | STRING | 과정의 전체 이름 |
| `department` | STRING | 과정을 제공하는 학과 |
| `credits` | INT | 학사 학점 수 |

> 🤖 **Genie Code 팁:** 질문하세요:
> *"Databricks Unity Catalog에서 기본 키 제약 조건이 있는 과정 테이블에 대한 CREATE TABLE 문을 작성합니다."*

In [ ]:
%sql
CREATE TABLE IF NOT EXISTS edu_dev.silver.courses (
  course_id   BIGINT NOT NULL,
  course_name STRING,
  department  STRING,
  credits     INT,
  CONSTRAINT courses_pk PRIMARY KEY (course_id)
);

### 작업 2.3 — 외래 키가 있는 `enrollments` 테이블 생성

어느 학생이 어느 과정에 등록되어 있는지 기록하는 `edu_dev.silver.enrollments`를 만듭니다. 이 테이블은 외래 키 제약 조건을 통해 `students` 및 `courses`를 참조해야 합니다.

| 열 | 유형 | 설명 |
|--------|------|-------------|
| `enrollment_id` | BIGINT | 고유 등록 레코드 (기본 키) |
| `student_id` | BIGINT | `students.student_id` 참조 |
| `course_id` | BIGINT | `courses.course_id` 참조 |
| `semester` | STRING | 학기 (예: `Spring 2023`) |
| `grade` | DECIMAL(4,2) | 0.0–10.0 스케일의 수치 등급 |

> 🤖 **Genie Code 팁:** 질문하세요:
> *"Databricks Unity Catalog에서 테이블을 생성할 때 다른 테이블을 참조하는 외래 키 제약 조건을 추가하는 방법은?"*

**힌트:** 각 외래 키에 대해 `CONSTRAINT <name> FOREIGN KEY (<col>) REFERENCES <catalog>.<schema>.<table>(<col>)`을 사용합니다.

In [ ]:
%sql
CREATE TABLE IF NOT EXISTS edu_dev.silver.enrollments (
  enrollment_id BIGINT NOT NULL,
  student_id    BIGINT,
  course_id     BIGINT,
  semester      STRING,
  grade         DECIMAL(4,2),
  CONSTRAINT enrollments_pk         PRIMARY KEY (enrollment_id),
  CONSTRAINT enrollments_students_fk FOREIGN KEY (student_id) REFERENCES edu_dev.silver.students(student_id),
  CONSTRAINT enrollments_courses_fk  FOREIGN KEY (course_id)  REFERENCES edu_dev.silver.courses(course_id)
);

### 샘플 데이터 — 이 셀을 실행하세요

다음 셀은 세 개의 테이블에 샘플 데이터를 삽입합니다. **연습 3을 계속하기 전에 수정 없이 실행하세요.**

In [ ]:
%sql
-- Populate students
INSERT INTO edu_dev.silver.students VALUES
(1001, 'Emma',   'Watson',    'emma.watson@lakeside.edu',    2022, 'Computer Science'),
(1002, 'Liam',   'Ahmed',     'liam.ahmed@lakeside.edu',     2023, 'Data Science'),
(1003, 'Sofia',  'Chen',      'sofia.chen@lakeside.edu',     2021, 'Mathematics'),
(1004, 'James',  'Okonkwo',   'james.okonkwo@lakeside.edu',  2022, 'Computer Science'),
(1005, 'Aisha',  'Patel',     'aisha.patel@lakeside.edu',    2023, 'Data Science'),
(1006, 'Noah',   'Yamamoto',  'noah.yamamoto@lakeside.edu',  2020, 'Mathematics'),
(1007, 'Olivia', 'Kowalski',  'olivia.kowalski@lakeside.edu',2021, 'Computer Science'),
(1008, 'Carlos', 'Reyes',     'carlos.reyes@lakeside.edu',   2022, 'Data Science'),
(1009, 'Mei',    'Lindqvist', 'mei.lindqvist@lakeside.edu',  2023, 'Mathematics'),
(1010, 'Jake',   'OBrien',    'jake.obrien@lakeside.edu',    2020, 'Computer Science');

-- Populate courses
INSERT INTO edu_dev.silver.courses VALUES
(101, 'Introduction to Programming',   'Computer Science', 3),
(102, 'Data Structures',               'Computer Science', 4),
(103, 'Statistics for Data Science',   'Data Science',     3),
(104, 'Machine Learning Fundamentals', 'Data Science',     4),
(105, 'Calculus I',                    'Mathematics',      4),
(106, 'Linear Algebra',                'Mathematics',      3);

-- Populate enrollments
INSERT INTO edu_dev.silver.enrollments VALUES
(1,  1001, 101, 'Spring 2023', 8.50),
(2,  1001, 102, 'Fall 2023',   7.80),
(3,  1002, 103, 'Spring 2023', 9.20),
(4,  1002, 104, 'Fall 2023',   6.50),
(5,  1003, 105, 'Spring 2022', 7.00),
(6,  1003, 106, 'Fall 2022',   8.00),
(7,  1004, 101, 'Fall 2022',   6.80),
(8,  1005, 103, 'Spring 2023', 9.00),
(9,  1006, 105, 'Fall 2020',   7.50),
(10, 1007, 102, 'Spring 2022', 8.20),
(11, 1008, 104, 'Fall 2023',   7.30),
(12, 1009, 106, 'Spring 2023', 8.80),
(13, 1010, 101, 'Fall 2021',   5.50),
(14, 1004, 102, 'Spring 2023', 7.10),
(15, 1005, 104, 'Fall 2023',   8.90);